In [2]:
#Data Handling 
import pandas as pd
import numpy as np

#Text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#Feature engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#Model building
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shivaprasad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shivaprasad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

**Data Understanding**


In [10]:
import pandas as pd

# Load dataset
df = pd.read_csv(r"C:\Users\shivaprasad\Downloads\twitter_training.csv\twitter_training.csv")
df


,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [12]:
# Sample data
print("\nSample data:")
print(df.head())




Sample data:
   2401  Borderlands  Positive  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

  im getting on borderlands and i will murder you all ,  
0  I am coming to the borders and I will kill you...     
1  im getting on borderlands and i will kill you ...     
2  im coming on borderlands and i will murder you...     
3  im getting on borderlands 2 and i will murder ...     
4  im getting into borderlands and i can murder y...     


In [13]:
# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns)



Shape: (74681, 4)

Columns: Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [14]:
# Assign proper column names
df.columns = ['id', 'entity', 'sentiment', 'text']

In [15]:
# Class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())




Class Distribution:
sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [16]:
# Check missing values
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
id             0
entity         0
sentiment      0
text         686
dtype: int64


In [17]:
# Remove missing values
df.dropna(inplace=True)
# Keep only needed columns
df = df[['text', 'sentiment']]

# Reset index
df.reset_index(drop=True, inplace=True)

In [18]:
print(df.shape)
print(df.columns)
print(df.head())
print(df['sentiment'].value_counts())

(73995, 2)
Index(['text', 'sentiment'], dtype='object')
                                                text sentiment
0  I am coming to the borders and I will kill you...  Positive
1  im getting on borderlands and i will kill you ...  Positive
2  im coming on borderlands and i will murder you...  Positive
3  im getting on borderlands 2 and i will murder ...  Positive
4  im getting into borderlands and i can murder y...  Positive
sentiment
Negative      22358
Positive      20654
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64


**NLP Preprocessing**


In [19]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


**Preprocessing Code**

In [23]:
# Initialize
stop_words = set(stopwords.words('english'))

# Keep important negation words
if 'not' in stop_words:
    stop_words.remove('not')

lemmatizer = WordNetLemmatizer()

# -----------------------------
# Clean Text Function
# -----------------------------
def clean_text(text):
    
    text = str(text).lower()  # lowercase
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    
    # Remove special characters & numbers
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords
    words = [w for w in words if w not in stop_words]
    
    # Lemmatization
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

# -----------------------------
# Apply safely
# -----------------------------
# Ensure clean dataframe
df = df[['text', 'sentiment']].copy()

# Handle missing values
df.dropna(subset=['text'], inplace=True)

# Apply preprocessing
df['clean_text'] = df['text'].apply(clean_text)

# Preview
print(df[['text', 'clean_text']].head())

                                                text  \
0  I am coming to the borders and I will kill you...   
1  im getting on borderlands and i will kill you ...   
2  im coming on borderlands and i will murder you...   
3  im getting on borderlands 2 and i will murder ...   
4  im getting into borderlands and i can murder y...   

                     clean_text  
0            coming border kill  
1    im getting borderland kill  
2   im coming borderland murder  
3  im getting borderland murder  
4  im getting borderland murder  


**Feature Engineering**

**Bag of Words**

In [24]:
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])
y = df['sentiment']

**TF-IDF**

In [25]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

**Train-Test Split**

In [26]:

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

**Model Building**

**Logistic Regression**

In [27]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

**Naive Bayes**

In [28]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

**Decision Tree**

In [29]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

DecisionTreeClassifier()

**Model Evaluation**

In [30]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, name):
    y_pred = model.predict(X_test)
    
    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

evaluate(lr, "Logistic Regression")
evaluate(nb, "Naive Bayes")
evaluate(dt, "Decision Tree")


 Logistic Regression
Accuracy: 0.6905872018379621
              precision    recall  f1-score   support

  Irrelevant       0.69      0.50      0.58      2624
    Negative       0.69      0.80      0.74      4463
     Neutral       0.69      0.64      0.67      3589
    Positive       0.69      0.74      0.71      4123

    accuracy                           0.69     14799
   macro avg       0.69      0.67      0.68     14799
weighted avg       0.69      0.69      0.69     14799


 Naive Bayes
Accuracy: 0.6451787282924522
              precision    recall  f1-score   support

  Irrelevant       0.75      0.35      0.48      2624
    Negative       0.62      0.81      0.70      4463
     Neutral       0.68      0.54      0.60      3589
    Positive       0.63      0.74      0.68      4123

    accuracy                           0.65     14799
   macro avg       0.67      0.61      0.62     14799
weighted avg       0.66      0.65      0.63     14799


 Decision Tree
Accuracy: 0.78349888